# 04 · Ablation study & tuning

The payoff of the modular design: a full ablation is one call. `build_ablation_suite` returns the standard ladder — the exact baseline, each of the eight LMGEC extensions added **alone**, and a **full** model combining them — as named `LMGECplus` instances ready to drop into `Experiment`.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=4.0, seed=0):
    """A small, well-separated multi-view dataset: (Xs, As, labels).

    Each view is Gaussian blobs around k cluster centres, plus a sparse
    intra-cluster adjacency so graph propagation has something to smooth.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d in dims:
        centres = rng.randn(k, d) * sep
        X = centres[labels] + rng.randn(n, d)
        Xs.append(X.astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            nbrs = rng.choice(same, size=min(6, len(same)), replace=False)
            A[i, nbrs] = 1.0
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from pymv.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## The default extension ladder

In [ ]:
from pymv.models.lmgecplus import build_ablation_suite, DEFAULT_ABLATIONS

for name, overrides in DEFAULT_ABLATIONS.items():
    print(f'{name:22s} {overrides}')

## Run the ablation with `Experiment`

In [ ]:
from pymv.experiment import Experiment
from pymv.metrics import clustering_accuracy, nmi_score, ari_score

suite = build_ablation_suite(n_clusters=k, embedding_dim=16, max_iter=30, random_state=0)

result = Experiment(
    datasets={'synthetic': (Xs, As, labels)},
    models=list(suite.values()),
    metrics=[clustering_accuracy, nmi_score, ari_score],
    runs=3,
).run()

Each row is one variant; comparing against `baseline` isolates the marginal effect of every extension. To ablate on a **real** dataset, pass `datasets=['acm']` (or any name from notebook 01) instead of the synthetic tuple.

## Restrict to a subset

`include=` picks specific variants — e.g. only the loss and clustering extensions.

In [ ]:
subset = build_ablation_suite(
    n_clusters=k, embedding_dim=16, max_iter=30, random_state=0,
    include=['baseline', 'E1 robust-l21', 'E7 soft-clustering', 'full'],
)
print('variants:', list(subset))

## Tune a variant with random search

Once an extension looks promising, `RandomSearchExperiment` sweeps its hyper-parameters. Component choices are just parameters, so you can search over them too.

In [ ]:
from pymv.experiment import RandomSearchExperiment
from pymv.models import LMGECplus

rs = RandomSearchExperiment(
    model_class=LMGECplus,
    param_distributions={
        'n_clusters': [k],
        'embedding_dim': [8, 16, 32],
        'temperature': [0.5, 1.0, 5.0],
        'reconstruction_loss': ['l2', 'l21', 'huber'],
        'clustering': ['kmeans', 'soft'],
        'max_iter': [30],
    },
    dataset=('synthetic', (Xs, As, labels)),
    metrics=[clustering_accuracy, nmi_score],
    best_metric='clustering_accuracy',
    n_iter=12,
    seed=0,
)
rs.run()
print('best params:', rs.result.best_trial['params'])

### Saving a report

Pass `save_markdown=True, experiment_name="my_ablation"` to either runner to write a full report (Markdown + `results.json` + `metrics.csv` + a reproducible `config.yaml`) under `experiments/<name>/`.